# Charmonia CNM Components: RHIC d+Au 200 GeV

This notebook analyzes the individual contribution of **Energy Loss** and **pT Broadening** components to the nuclear modification factor $R_{dAu}$ for Charmonia (J/psi) at $\sqrt{s_{NN}} = 200$ GeV.

It uses the `CNMCombine` class to load physics parameters and `eloss_cronin_centrality` for visualization.

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add parent directory to path to import modules
sys.path.append("..")

from cnm_combine.cnm_combine import CNMCombine
from eloss_code.eloss_cronin_centrality import (
    plot_RpA_vs_y_components_per_centrality,
    plot_RpA_vs_pT_components_per_centrality,
    plot_RpA_vs_centrality_components_band,
    set_publication_style
)

# Apply generic publication style
set_publication_style()

ModuleNotFoundError: No module named 'cnm_combine'

In [ ]:
# Initialize CNM Analysis for RHIC d+Au 200 GeV
cnm = CNMCombine.from_defaults("200")

# Extract underlying physics objects for component plotting
P = cnm.P          # Particle (J/psi)
roots = cnm.roots  # 200 GeV
qp = cnm.qp        # Quench Params (q0, broad inputs)
gl = cnm.gl        # Glauber (d+Au)

# Output directory
OUTDIR = Path("../outputs/RHIC/CNM_200_Components")
OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded System: {cnm.spec.system}")
print(f"Output Directory: {OUTDIR}")

[OpticalGlauber] Target A=197, d=0.535 fm, σ_nn=42.00 mb
[OpticalGlauber] ∫ d²s T_A(s) ≈ 196.792  (should be ~A=197)
[OpticalGlauber] ∫ d²s T_d(s) ≈ 1.9972  (should be ~2)
[OpticalGlauber] Tabulating overlaps T_AA(b), T_pA(b), T_dA(b)...
[OpticalGlauber] σ_tot: AA=7050.0 mb, pA=1699.6 mb, dA=2112.7 mb
Loaded System: dA
Output Directory: ../outputs/RHIC/CNM_200_Components


## 1. Components vs Rapidity ($y$)

We investigate how Energy Loss vs. Broadening varies with rapidity for different centrality classes.

In [ ]:
y_axis = np.linspace(-3.0, 3.0, 61)
pT_avg = 0.0 # pT integrated-ish or low pT for y-dependence

# Centrality Bins for d+Au
# RHIC experiments often use: 0-20, 20-40, 40-60, 60-88
cent_bins = [(0, 20), (20, 40), (40, 60), (60, 88)]

fig, axes = plot_RpA_vs_y_components_per_centrality(
    P, roots, qp, gl,
    cent_bins,
    y_axis, pT_avg,
    show_components=("loss", "broad", "total", "npdf", "cnm"),
    q0_pair=cnm.q0_pair,
    p0_scale_pair=(0.9, 1.1),
    ncols=2,
    suptitle=r"$J/\psi$ d+Au 200 GeV: Components vs $y$"
)

plt.savefig(OUTDIR / "RpA_components_vs_y_dAu200.pdf", bbox_inches="tight")
plt.show()

## 2. Components vs Transverse Momentum ($p_T$)

We look at the $p_T$ dependence in three rapidity windows: Backward, Mid, and Forward.

In [ ]:
pT_edges = np.linspace(0, 10, 21) # 0 to 10 GeV

# Standard RHIC Rapidity Windows
rapidity_wins = [
    (-2.2, -1.2, "Backward"),
    (-0.35, 0.35, "Mid"),
    (1.2, 2.2, "Forward")
]

# Plot for each window
for y0, y1, label in rapidity_wins:
    fig, axes = plot_RpA_vs_pT_components_per_centrality(
        P, roots, qp, gl,
        cent_bins,
        pT_edges, (y0, y1),
        show_components=("loss", "broad", "total", "npdf", "cnm"),
        q0_pair=cnm.q0_pair,
        p0_scale_pair=(0.9, 1.1),
        ncols=4, # 4 centrality bins, so 4 columns fits well
        step=True,
        suptitle=rf"$J/\psi$ d+Au 200 GeV ({label}): Components vs $p_T$"
    )
    
    plt.savefig(OUTDIR / f"RpA_components_vs_pT_dAu200_{label}.pdf", bbox_inches="tight")
    plt.show()

## 3. Components vs Centrality (Integrated Band)

Finally, we integrate over $p_T$ and look at the centrality dependence.

In [ ]:
pT_integ = (0.0, 5.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for i, (y0, y1, label) in enumerate(rapidity_wins):
    ax = axes[i]
    
    # Calculate bands
    (labels, RL_c, RL_lo, RL_hi, RB_c, RB_lo, RB_hi, 
     RT_c, RT_lo, RT_hi, 
     RMB_loss, RMB_broad, RMB_tot) = cnm._calc_eloss_broad_band_vs_centrality(
         cent_bins, (y0, y1), pT_integ
     )
    
    # Note: accessing private method _calc_eloss... is okay for detailed notebook, 
    # but we can also use the helper function directly if we imported it.
    # Actually, let's use the plot helper which calculates internally if we don't pass data,
    # BUT `plot_RpA_vs_centrality_components_band` expects PRE-CALCULATED data arrays.
    # So we used the CNMCombine helper or the rpa_band_vs_centrality function.
    # Let's import the calculation function to be explicit.
    
    from eloss_code.eloss_cronin_centrality import rpa_band_vs_centrality
    
    (labels_calc,
     RL_c, RL_lo, RL_hi,
     RB_c, RB_lo, RB_hi,
     RT_c, RT_lo, RT_hi,
     RMB_loss, RMB_broad, RMB_tot) = rpa_band_vs_centrality(
         P, roots, qp, gl,
         cent_bins, (y0, y1), pT_integ,
         q0_pair=cnm.q0_pair,
         p0_scale_pair=(0.9, 1.1),
         weight_kind="pp"
     )

    plot_RpA_vs_centrality_components_band(
        cent_bins, labels_calc,
        RL_c, RL_lo, RL_hi, RMB_loss,
        RB_c, RB_lo, RB_hi, RMB_broad,
        RT_c, RT_lo, RT_hi, RMB_tot,
        show=("loss", "broad", "total", "npdf", "cnm"),
        ax=ax,
        ylabel=r"$R_{dAu}$",
        note=f"{label}\n{y0}<y<{y1}",
        system_label="d+Au 200 GeV"
    )

plt.savefig(OUTDIR / "RpA_components_vs_cent_dAu200.pdf", bbox_inches="tight")
plt.show()